In [ ]:
from expression import build_me_model
from uniform_processes import biomass
from utils import parameters as params
import copy
import pandas as pd
import numpy as np
from tqdm import tqdm
import multiprocessing
from core.reaction import ME_Reaction


def get_mod(frac):
    store = dict()
    m_id = 'dummy_{}'.format(frac) if frac is not None else 'no_dummy'

    model, builder = build_me_model.build_me(non_machinery = [], minimal_proteome = True, 
                   compress_mrna = False, unmodeled_protein_frac = frac, model_id = m_id)
    
    bad_rxn = list()
    for r in params.human_model.reactions:
        if len(r.genes) == 0:
            for r_ in model.reactions: 
                if isinstance(r_, ME_Reaction):
                    if (r_.id == r.id) or (r_.cobra_id == r.id):
                        if len({k for k,v in r_.coupled_metabolites.items() if 'HGNC:DUMMY' not in k.id}) > 0:
                            bad_rxn.append(r_.id)
                        break
    if len(bad_rxn) > 0
    

    models = [model]
    if frac is not None:
        sink_model = copy.deepcopy(model)
        sink_model.id = 'dummy_sink_{}'.format(frac)
        sink_model.add_boundary(sink_model.metabolites.get_by_id('HGNC:10419_folded_protein_c'), type = 'sink')
        models.append(sink_model)

    for model in models:
        store[model.id] = {'model': model, 'builder': builder}

        print('Solve ' + m_id)
        sln, stat, _ = store[model.id]['model'].solve_lp(mu_val = 1e-9)
        ir = store[model.id]['model'].infeasible_reactions(1e-9, sln, stat)

        store[model.id]['sln'] = sln
        store[model.id]['stat'] = stat
        store[model.id]['infeasible_reactions'] = ir

    return store

def get_mods(fracs, n_cores = 3):
    pool = multiprocessing.Pool(processes = n_cores)
    stores = pool.map(get_mod,fracs)
    pool.close()
    return stores

In [ ]:
stores = get_mods(fracs = [None, 0.001, params.unmodeled_protein_frac])

res = dict()
for store in stores:
    for k,v in store.items():
        res[k] = v
print(list(res.keys()))

In [ ]:
tol = np.max([abs(v) for v in res['no_dummy']['infeasible_reactions'].values()])

r_ids = []
for k_ in res:
    r_ids += [k for k,v in res[k_]['infeasible_reactions'].items() if abs(v) > tol]

r_ids = pd.Series([biomass.pb_reaction.id] + [r.id for r in biomass.biomass_reactions]  + r_ids).unique().tolist()
res_df = pd.DataFrame(columns = res.keys(), index = r_ids)    

for k in res.keys():
    for r_id in r_ids:
        try:
            res_df.loc[r_id, k] = res[k]['sln'][res[k]['model'].reactions.index(r_id)]
        except:
            res_df.loc[r_id, k] = float('nan')

fail = res_df[abs(res_df['dummy_0.879584658137385'] - res_df['dummy_sink_0.879584658137385']) > tol]

In [ ]:
model = res['no_dummy']['model']

In [ ]:
self = res['no_dummy']['model']
self.m_model = params.human_model.copy()
mismatch = check_coupling(self)